# Pipeline Diagnostics — Sentinel-1 Preprocessing

This notebook verifies each stage of the SNAP preprocessing pipeline by 
reading the output GeoTIFFs and inspecting raster statistics. 

## Methodology

Pipeline operators were added incrementally and outputs verified at each stage. Due to a problem - invalid data (e.g., all-zero rasters).

## Pipeline stages tested

| Test | Operators | Expected output |
|---|---|---|
| 1 | Read + Subset | Intensity_VV, raw DN values 1-15000 |
| 2 | + Calibration | Sigma0_VV linear, range 0.0001-100 |
| 3 | + Speckle-Filter (Refined Lee 5×5) | Smoothed sigma0, similar range |
| 4 | + Terrain-Correction (SRTM 1Sec HGT) | EPSG:4326 raster (failed)|
| 5 | + Terrain-Correction (Copernicus 30m) | EPSG:4326 raster (working) |

In [1]:
import rasterio
import numpy as np
from pathlib import Path

OUTPUT_DIR = Path(r'D:\TexasBeryl\output')

def inspect_raster(path: Path, label: str):
    """Print summary statistics for a SNAP-processed GeoTIFF."""
    print(f"\n{'='*60}")
    print(f"{label}")
    print(f"File: {path.name}")
    print(f"{'='*60}")
    
    if not path.exists():
        print("FILE DOES NOT EXIST")
        return
    
    with rasterio.open(path) as src:
        print(f"CRS:    {src.crs}")
        print(f"Shape:  {src.shape}")
        print(f"Bounds: {src.bounds}")
        
        data = src.read(1)
        valid = data[(data != 0) & ~np.isnan(data)]
        
        print(f"\nNon-zero non-NaN pixels: {len(valid):,} / {data.size:,} "
              f"({100*len(valid)/data.size:.1f}%)")
        
        if len(valid) > 0:
            print(f"  Min:        {valid.min():.6f}")
            print(f"  Max:        {valid.max():.4f}")
            print(f"  Mean:       {valid.mean():.4f}")
            print(f"  Percentiles 1/50/99: {np.percentile(valid, [1,50,99])}")
        else:
            print("  ALL PIXELS ARE ZERO OR NaN — operator failed silently")

## Test 1: Read + Subset

Verifies that the geographic subset correctly extracts the Galveston AOI 
from the full Sentinel-1 scene. Output should be raw Intensity_VV (DN values).

In [14]:
inspect_raster(OUTPUT_DIR / 'test1_subset_only.tif', 'Test 1 — Read + Subset')


Test 1 — Read + Subset
File: test1_subset_only.tif
CRS:    None
Shape:  (4146, 5846)
Bounds: BoundingBox(left=0.0, bottom=4146.0, right=5846.0, top=0.0)

Non-zero non-NaN pixels: 24,237,516 / 24,237,516 (100.0%)
  Min:        7.000000
  Max:        14847.0000
  Mean:       106.2644
  Percentiles 1/50/99: [ 25.  86. 315.]


## Test 2: + Calibration

Adds radiometric calibration. Output is sigma0 in linear scale 
(typical range 0.0001 – 100).

In [15]:
inspect_raster(OUTPUT_DIR / 'test2_calibrated.tif', 'Test 2 — + Calibration')



Test 2 — + Calibration
File: test2_calibrated.tif
CRS:    None
Shape:  (4146, 5846)
Bounds: BoundingBox(left=0.0, bottom=4146.0, right=5846.0, top=0.0)

Non-zero non-NaN pixels: 24,237,516 / 24,237,516 (100.0%)
  Min:        0.000150
  Max:        665.2210
  Mean:       0.0563
  Percentiles 1/50/99: [0.00195218 0.02239542 0.30185725]


## Test 3: + Speckle-Filter (Refined Lee 5×5)

Adds speckle filtering. Output should retain similar statistics to Test 2 
but with reduced extreme values (filter clips outliers).

In [16]:
inspect_raster(OUTPUT_DIR / 'test3_speckle.tif', 'Test 3 — + Speckle-Filter')


Test 3 — + Speckle-Filter
File: test3_speckle.tif
CRS:    None
Shape:  (4146, 5846)
Bounds: BoundingBox(left=0.0, bottom=4146.0, right=5846.0, top=0.0)

Non-zero non-NaN pixels: 24,237,516 / 24,237,516 (100.0%)
  Min:        0.001077
  Max:        200.4325
  Mean:       0.0516
  Percentiles 1/50/99: [0.00354623 0.02893831 0.24623474]


## Test 4: + Terrain-Correction (SRTM 1Sec HGT) — FAILED

!!! This test produced all-zero output despite reporting success.!!!

Hypothesis: SRTM 1Sec HGT auto-download in SNAP 13 fails silently 
(the operator finishes without error but writes zero values for all pixels).

In [18]:
inspect_raster(OUTPUT_DIR / 'test4_terrain.tif', 'Test 4 — Terrain-Correction with SRTM ')



Test 4 — Terrain-Correction with SRTM (FAILED)
File: test4_terrain.tif
CRS:    EPSG:4326
Shape:  (2580, 3712)
Bounds: BoundingBox(left=-95.40933288515026, bottom=28.967986151207388, right=-94.74242361821993, top=29.43151683781306)

Non-zero non-NaN pixels: 0 / 9,576,960 (0.0%)
  ALL PIXELS ARE ZERO OR NaN — operator failed silently


## Test 5: + Terrain-Correction (Copernicus 30m Global DEM) — FAILED

!!! This test produced all-zero output despite reporting success.!!!

Switching the DEM to Copernicus 30m Global DEM (also auto-download, but 
hosted on ESA infrastructure) resolves the issue.

In [19]:
inspect_raster(OUTPUT_DIR / 'test5_terrain_copernicus.tif', 'Test 5 — Terrain-Correction with Copernicus DEM')



Test 5 — Terrain-Correction with Copernicus DEM
File: test5_terrain_copernicus.tif
CRS:    EPSG:4326
Shape:  (2580, 3712)
Bounds: BoundingBox(left=-95.40933288515026, bottom=28.967986151207388, right=-94.74242361821993, top=29.43151683781306)

Non-zero non-NaN pixels: 0 / 9,576,960 (0.0%)
  ALL PIXELS ARE ZERO OR NaN — operator failed silently


## Manual analysis
To isolate the failure, Terrain-Correction was invoked manually in SNAP 
on the saved, working `test3_speckle.tif` — bypassing Graph Builder entirely.
This gave an error:
"Input should be a SAR product"
Conclusion:
Terrain-Correction requires a complete SAR product with full radar metadata, not GEOTIFF format.
### 3 Other issues revealed while manual testing
1.Missing Apply-Orbit-File operator.
2.Subset destroys SAR metadata when applied early.
3.GeoTIFF cannot be re-fed into Terrain-Correction.
## Fix
The corrected pipeline (`flood_pipeline.xml`) integrates these findings:




In [6]:
inspect_raster(OUTPUT_DIR / 'galveston_before_VV_dB.tif', 'FULL PIPELINE — Galveston Before')
inspect_raster(OUTPUT_DIR / 'galveston_after_VV_dB.tif', 'FULL PIPELINE — Galveston Before')



FULL PIPELINE — Galveston Before
File: galveston_before_VV_dB.tif
CRS:    EPSG:4326
Shape:  (1671, 3063)
Bounds: BoundingBox(left=-95.35016162335023, bottom=29.04992775409342, right=-94.79985368029861, top=29.350144722046164)

Non-zero non-NaN pixels: 5,118,273 / 5,118,273 (100.0%)
  Min:        -28.453047
  Max:        22.7817
  Mean:       -16.4561
  Percentiles 1/50/99: [-24.23664093 -15.33829212  -6.73745483]

FULL PIPELINE — Galveston Before
File: galveston_after_VV_dB.tif
CRS:    EPSG:4326
Shape:  (1671, 3062)
Bounds: BoundingBox(left=-95.35007539097082, bottom=29.049891514271305, right=-94.79994711097602, top=29.350108482224048)

Non-zero non-NaN pixels: 5,116,602 / 5,116,602 (100.0%)
  Min:        -33.444031
  Max:        22.1571
  Mean:       -17.0212
  Percentiles 1/50/99: [-29.40983959 -14.55903625  -6.01339036]
